In [1]:
import geopandas as gpd
import folium
from shapely import wkt
import osmnx as ox

In [2]:
# street network 

G = ox.graph_from_place(
  'Christchurch, New Zealand',
  network_type='drive',
  simplify=True,
)

G = ox.project_graph(G, to_crs=2193)

edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
nodes = ox.graph_to_gdfs(G, nodes=True, edges=False)

In [3]:
property = gpd.read_file('output/property.gpkg', engine='pyogrio')
reach_200 = gpd.read_file('output/property_reach_200m.gpkg', engine='pyogrio')
access_points = gpd.read_file('output/property_accesspoints.gpkg', engine='pyogrio')

In [4]:
access_points = access_points.rename(columns={'geometry': 'access_point'})
reach_200 = reach_200.rename(columns={'geometry': 'reach_200m'})

In [5]:
property_full = property.merge(
    reach_200[['property_id', 'reach_200m']],
    on='property_id',
    how='left'
)

property_full = property_full.merge(
    access_points[['property_id', 'access_point']],
    on='property_id',
    how='left'
)

In [17]:
sample_size=100

In [18]:
sub_property = property_full.sample(n=sample_size, random_state=1).copy()

property_wgs = sub_property.to_crs(epsg=4326)

property_wgs["access_point"] = gpd.GeoSeries(
    sub_property["access_point"],
    crs=property_full.crs
).to_crs(epsg=4326)

property_wgs["reach_200m"] = gpd.GeoSeries(
    sub_property["reach_200m"],
    crs=property_full.crs
).to_crs(epsg=4326)

In [19]:
# edges_wgs = edges.to_crs(epsg=4326)
# nodes_wgs = nodes.to_crs(epsg=4326)

m = folium.Map(
  location=[property_wgs.geometry.y.mean(), property_wgs.geometry.x.mean()],
  zoom_start=14,
  tiles='CartoDB positron'
)

# folium.GeoJson(
#     edges_wgs,
#     name="Street Network",
#     style_function=lambda x: {
#         "color": "gray",
#         "weight": 1,
#         "opacity": 0.5
#     }
# ).add_to(m)

# folium.GeoJson(
#     nodes_wgs,
#     name="Street Nodes",
#     marker=folium.CircleMarker(
#         radius=1,
#         color="black",
#         fill=True,
#         fill_opacity=0.7
#     )
# ).add_to(m)

for idx, row in property_wgs.iterrows():
  # Property point
    folium.CircleMarker(
        [row.geometry.y, row.geometry.x],
        radius=6,
        color="purple",
        fill=True,
        fill_opacity=1,
        popup=f"Property ID: {idx}"
    ).add_to(m)

    # Access point
    if row["access_point"] is not None:
        folium.CircleMarker(
            [row["access_point"].y, row["access_point"].x],
            radius=4,
            color="blue",
            fill=True,
            fill_opacity=0.9,
            popup="Access point"
        ).add_to(m)

    # Reachable streets (200 m)
    if row["reach_200m"] is not None and not row["reach_200m"].is_empty:
        folium.GeoJson(
            row["reach_200m"],
            name="Reach 200m",
            style_function=lambda x: {
                "color": "red",
                "weight": 3,
                "opacity": 0.6
            }
        ).add_to(m)

m